# WasteLess Recommender
Beveelt recepten aan op basis van wat een gebruiker thuis heeft, met prioriteit voor ingrediënten die binnenkort verlopen.

**Zorg dat de volgende bestanden in dezelfde map staan:**
- `recepten_AH.xlsx`
- `pantry_proxies.xlsx`

## 1. Installeer afhankelijkheden

In [1]:
%pip install pandas numpy openpyxl rapidfuzz scikit-learn nltk --quiet

import nltk
nltk.download('punkt', quiet=True)

Note: you may need to restart the kernel to use updated packages.


True

## 2. Imports & configuratie

In [2]:
import argparse, re
from datetime import datetime

import numpy as np
import pandas as pd
from nltk.stem.snowball import SnowballStemmer as _Stemmer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

_stemmer = _Stemmer("dutch")

# ── Bestandspaden ──────────────────────────────────────────────────────
RECIPES_FILE = "recepten_AH.xlsx"
PANTRY_FILE  = "pantry_proxies.xlsx"

# ── Scoring-gewichten ──────────────────────────────────────────────────
WEIGHT_COVERAGE  = 0.75
WEIGHT_TFIDF     = 0.25
WASTE_BONUS_MAX  = 0.35
RATING_BOOST     = 0.20
RATING_MAX       = 10.0

# ── Lengte-penalty ─────────────────────────────────────────────────────
MIN_INGREDIENTS  = 6
PENALTY_STRENGTH = 0.40

# ── Andere drempelwaarden ──────────────────────────────────────────────
ALMOST_DONE_RATIO = 0.60
MAX_PER_CATEGORY  = 3
MMR_LAMBDA        = 0.7

UNIT_TO_BASE = {
    "kg": 1000, "g": 1, "mg": 0.001, "l": 1000, "ml": 1, "dl": 100,
    "el": 15, "tl": 5, "eetlepel": 15, "theelepel": 5,
    "stuks": 100, "stuk": 100, "teen": 5, "takje": 5, "snuf": 2, "snufje": 2,
}

STOP_WORDS = {
    "verse", "vers", "gedroogde", "gedroogd", "gerookte", "gerookt",
    "gesneden", "gehakte", "gehakt", "gekookte", "gekookt", "halfvolle",
    "volle", "magere", "milde", "zoete", "zoet", "grote", "groot",
    "kleine", "klein", "biologische", "biologisch", "ah", "mix",
    "voor", "van", "met", "en", "of", "pure", "belegen", "geraspte",
    "italiaanse", "italiaans", "spaanse", "spaans", "franse", "grieks", "griekse",
    "droge", "soepele", "robuuste", "vegan", "plantaardige", "traditionele",
    "houdbare", "diepvries", "premium", "classic", "original",
}

CATEGORY_KEYWORDS = {
    "ontbijt":     ["wafel", "broodje", "bagel", "smoothie", "toast", "pancake"],
    "soep":        ["soep", "bouillon"],
    "salade":      ["salade", "sla"],
    "snack":       ["snack", "chips", "borrel", "dip", "crackers"],
    "dessert":     ["taart", "cake", "cookie", "muffin", "dessert", "pudding"],
    "pasta":       ["pasta", "spaghetti", "penne", "lasagne", "orzo"],
    "vlees":       ["kip", "gehakt", "biefstuk", "spekjes", "chorizo", "bacon"],
    "vis":         ["zalm", "kabeljauw", "tonijn", "garnalen", "makreel"],
    "vegetarisch": ["tofu", "halloumi", "kikkererwten", "linzen", "falafel"],
    "stamppot":    ["stamppot", "puree", "stoof"],
}

print("Configuratie geladen")

Configuratie geladen


## 3. Hulpfuncties

In [3]:
def detect_category(name):
    # Simpele rule-based categorisatie op basis van keywords in de receptnaam
    # → wordt later gebruikt voor diversificatie (categorie-cap)
    name = name.lower()
    for cat, kws in CATEGORY_KEYWORDS.items():
        if any(kw in name for kw in kws):
            return cat
    return "overig"


def normalize(text):
    # Normalisatiepipeline:
    # - lowercase + tokenisatie
    # - verwijderen van stopwoorden en korte tokens (ruis)
    # - stemming (Snowball) om varianten te reduceren tot dezelfde stam
    words = [w for w in re.split(r"\s+", text.lower().strip())
             if w not in STOP_WORDS and len(w) > 2]

    # Alleen eerste 3 tokens gebruiken om ruis en variatie te beperken
    stemmed = [_stemmer.stem(w) for w in words[:3]]

    return " ".join(stemmed)


def parse_ingredient(line):
    # Parse van ingrediëntenregel naar (genormaliseerde naam, hoeveelheid in basiseenheid)
    # Regex pakt: hoeveelheid + optionele unit + ingrediëntnaam
    m = re.match(r"^([\d.,½¼¾]+)\s*([a-zA-Z]+)?\s+(.+)$", line.lower().strip())

    if not m:
        # Fallback: geen hoeveelheid gevonden → standaardwaarde gebruiken
        return normalize(line), 100.0

    # Ondersteuning voor fractionele notaties (½, ¼, ¾)
    qty  = float(m.group(1).replace(",", ".").replace("½", "0.5")
                            .replace("¼", "0.25").replace("¾", "0.75"))

    unit = (m.group(2) or "stuks").lower()

    # Conversie naar uniforme basiseenheid (gram/ml/stuks-equivalent)
    # → maakt vergelijking tussen pantry en recept mogelijk
    return normalize(m.group(3)), qty * UNIT_TO_BASE.get(unit, 100)


def parse_date(value):
    # Datumparsing (ongeldige waarden → NaT)
    parsed = pd.to_datetime(value, errors="coerce", dayfirst=True)

    # Normaliseren naar datum (zonder tijd) voor consistente vergelijking
    return pd.NaT if pd.isna(parsed) else pd.Timestamp(parsed).normalize()


def urgency_weight(expiry):
    # Urgentiefunctie u(d) (zie verslag)
    # → verhoogt gewicht naarmate expiry nadert
    exp = parse_date(expiry)

    if pd.isna(exp):
        return 1.0

    days = (exp - pd.Timestamp(datetime.today()).normalize()).days

    # Discrete drempels voor urgency (steile toename vlak voordat houdbaarheidsdatum wordt overschreden)
    for threshold, weight in [(0, 2.2), (1, 2.0), (2, 1.8), (4, 1.6),
                               (7, 1.4), (14, 1.15), (30, 1.05)]:
        if days <= threshold:
            return weight

    return 1.0


print("Hulpfuncties geladen")

Hulpfuncties geladen


## 4. Voorraad & ingrediënt-matching

In [4]:
def build_pantry(pantry_df, user_id):
    pantry = {}

    # Bouw een geaggregeerde pantry per gebruiker (naam → hoeveelheid + expiry)
    for _, r in pantry_df[pantry_df["user_id"] == user_id].iterrows():
        name   = normalize(str(r["ingredient"]))

        # Conversie naar uniforme basiseenheid (consistent met recepten)
        qty    = float(r["hoeveelheid"]) * UNIT_TO_BASE.get(str(r["eenheid"]).lower(), 100)

        expiry = parse_date(r.get("expiry_date", pd.NaT))

        if name not in pantry:
            pantry[name] = {"quantity": qty, "expiry": expiry}
        else:
            # Zelfde ingrediënt → hoeveelheden optellen
            pantry[name]["quantity"] += qty

            # Bewaar de vroegste expiry (meest urgente)
            e = pantry[name]["expiry"]
            if pd.isna(e) or (not pd.isna(expiry) and expiry < e):
                pantry[name]["expiry"] = expiry

    # Urgentie één keer berekenen en opslaan (efficiënter dan per recept)
    for item in pantry.values():
        item["urgency"] = urgency_weight(item["expiry"])

    return pantry


def fuzzy_lookup(name, pantry):
    # Exacte lookup in genormaliseerde ruimte
    # → bewust geen fuzzy matching om false positives te vermijden (zie verslag)
    item = pantry.get(name)

    if item:
        return item["quantity"], item["urgency"]

    # Geen match → geen dekking, neutrale urgentie
    return 0.0, 1.0


def build_recipe_matrix(recipes_df):
    parsed_recipes = []
    unique_names   = set()

    # Parse alle recepten naar (ingredient, hoeveelheid)-paren
    for _, recipe in recipes_df.iterrows():
        lines  = [l.strip() for l in str(recipe.get("Ingrediënten", "")).splitlines() if l.strip()]
        parsed = [parse_ingredient(l) for l in lines]

        parsed_recipes.append(parsed)

        # Verzamel unieke ingrediënten voor snelle lookup
        unique_names.update(name for name, _ in parsed)

    return parsed_recipes, sorted(unique_names)


def match_pantry_to_ingredients(unique_names, pantry):
    # ingrediënt → (beschikbaar, urgentie)
    # → voorkomt herhaalde dictionary lookups binnen de hoofdloop
    return {name: fuzzy_lookup(name, pantry) for name in unique_names}


def vectorized_coverage(parsed_recipes, lookup):
    n = len(parsed_recipes)

    # Arrays voor vectorized scoring (sneller dan Python loops, zie verslag)
    scores         = np.zeros(n)
    full_counts    = np.zeros(n, dtype=int)
    partial_counts = np.zeros(n, dtype=int)
    totals         = np.zeros(n, dtype=int)
    urgent_counts  = np.zeros(n, dtype=int)

    for i, ingredients in enumerate(parsed_recipes):
        if not ingredients:
            continue

        coverages = []

        for name, needed in ingredients:
            available, urgency = lookup.get(name, (0.0, 1.0))

            # Coverage: verhouding beschikbaar / nodig
            raw = min(available, needed) / needed if needed > 0 else (1.0 if available else 0.0)

            # Urgentie-weighting + limiet op 1.8 om extreme waarden te voorkomen
            coverages.append(min(raw * urgency, 1.8))

            # Statistieken voor filtering en bonusberekening
            if urgency >= 1.6 and raw > 0:
                urgent_counts[i]  += 1
            if raw >= 1.0:
                full_counts[i]    += 1
            elif raw > 0:
                partial_counts[i] += 1

        totals[i] = len(coverages)

        # Gemiddelde coverage over alle ingrediënten
        scores[i] = np.mean(coverages) if coverages else 0.0

    return scores, full_counts, partial_counts, totals, urgent_counts


print("Coverage-functies geladen")

Coverage-functies geladen


## 5. Diversiteits-herrangschikking
MMR voorkomt dat de top-10 gevuld wordt met bijna identieke recepten. Het balanceert relevantie (score) tegen diversiteit (similariteit met al gekozen recepten) via lambda.

In [5]:
def mmr_rerank(candidates, tfidf_matrix, recipe_index, top_n, score_col="score_final"):
    selected, remaining = [], candidates.copy()
    while len(selected) < top_n and len(remaining):
        if not selected:
            best = remaining.iloc[0]
        else:
            sel_vecs = tfidf_matrix[[recipe_index[r["recipe_id"]] for r in selected]]
            mmr = []
            for _, row in remaining.iterrows():
                idx = recipe_index.get(row["recipe_id"])
                sim = float(cosine_similarity(tfidf_matrix[idx:idx+1], sel_vecs).max()) if idx is not None else 1.0
                mmr.append(MMR_LAMBDA * row[score_col] - (1 - MMR_LAMBDA) * sim)
            remaining = remaining.copy()
            remaining["_mmr"] = mmr
            best = remaining.loc[remaining["_mmr"].idxmax()]
        selected.append(best.to_dict())
        remaining = remaining[remaining["recipe_id"] != best["recipe_id"]]
    return pd.DataFrame(selected).drop(columns=["_mmr"], errors="ignore").reset_index(drop=True)


def cap_categories(df, top_n):
    counts, selected, reserve = {}, [], []
    for _, row in df.iterrows():
        cat = row.get("category", "overig")
        if counts.get(cat, 0) < MAX_PER_CATEGORY and len(selected) < top_n:
            selected.append(row); counts[cat] = counts.get(cat, 0) + 1
        else:
            reserve.append(row)
    for row in reserve:
        if len(selected) >= top_n: break
        selected.append(row)
    return pd.DataFrame(selected).reset_index(drop=True)


print("Diversiteits-functies geladen")

Diversiteits-functies geladen


## 6. WasteLess Recommender klasse
De aanbeveler is geïmplementeerd als een klasse zodat de data (recepten, pantry, 
TF-IDF matrix) eenmalig wordt ingeladen en hergebruikt voor alle gebruikers. Dit 
voorkomt dat bestanden en vectoren opnieuw worden berekend bij elke aanbeveling, 
wat de snelheid aanzienlijk verbetert. De klasse combineert pantry coverage, 
urgentieweging, TF-IDF similariteit en MMR-diversificatie in één pipeline, zoals 
beschreven in het verslag.

In [6]:
class WasteLessRecommender:

    def __init__(self, recipes_file=RECIPES_FILE, pantry_file=PANTRY_FILE):
        print("Data laden...", end=" ", flush=True)
        self.recipes = pd.read_excel(recipes_file)
        self.pantry  = pd.read_excel(pantry_file)
        if "expiry_date" in self.pantry.columns:
            self.pantry["expiry_date"] = self.pantry["expiry_date"].apply(parse_date)
        else:
            self.pantry["expiry_date"] = pd.NaT

        self.users = sorted(self.pantry["user_id"].unique())
        self.recipes["category"]      = self.recipes["Naam"].apply(detect_category)
        self.recipes["n_ingredients"] = self.recipes["Ingrediënten"].fillna("").apply(
            lambda x: len([l for l in x.splitlines() if l.strip()])
        )
        print(f"[OK] ({len(self.recipes)} recepten, {len(self.users)} gebruikers)")

        print("Recepten parsen...", end=" ", flush=True)
        self.parsed_recipes, self.unique_names = build_recipe_matrix(self.recipes)
        print("[OK]")

        self._build_tfidf()

    def _build_tfidf(self):
        print("TF-IDF bouwen...", end=" ", flush=True)
        recipe_docs = self.recipes["Ingrediënten"].fillna("").apply(
            lambda x: " ".join(normalize(l) for l in x.splitlines() if l.strip())
        ).tolist()
        user_docs = [
            " ".join(normalize(str(v)) for v in self.pantry[self.pantry["user_id"] == uid]["ingredient"])
            + " " + " ".join(self.pantry[self.pantry["user_id"] == uid]["categorie"].dropna().unique())
            for uid in self.users
        ]
        mat = TfidfVectorizer(ngram_range=(1, 2), min_df=1).fit_transform(recipe_docs + user_docs)
        self.recipe_vecs  = mat[:len(recipe_docs)]
        self.user_vecs    = mat[len(recipe_docs):]
        self.recipe_index = {rid: i for i, rid in enumerate(self.recipes["ID"].tolist())}
        print("[OK]")

    def recommend(self, user_id, top_n=10, only_almost_done=False, verbose=False):
        if user_id not in self.users:
            raise ValueError(f"Gebruiker {user_id} niet gevonden.")

        uid_idx   = list(self.users).index(user_id)
        profile   = self.pantry[self.pantry["user_id"] == user_id]["profiel"].iloc[0]
        pantry    = build_pantry(self.pantry, user_id)
        tfidf_sim = cosine_similarity(self.user_vecs[uid_idx:uid_idx+1], self.recipe_vecs)[0]

        lookup  = match_pantry_to_ingredients(self.unique_names, pantry)
        cov_scores, fulls, partials, totals, urgents = vectorized_coverage(self.parsed_recipes, lookup)

        n_ing   = self.recipes["n_ingredients"].values
        ratings = self.recipes["Gemiddelde rating"].fillna(5.0).astype(float).values

        penalties = np.where(
            (urgents >= 2) | (n_ing >= MIN_INGREDIENTS),
            1.0,
            1.0 - PENALTY_STRENGTH * (MIN_INGREDIENTS - n_ing) / MIN_INGREDIENTS
        )
        penalties = np.clip(penalties, 0.0, 1.0)

        waste_ratio = np.where(totals > 0, urgents / totals, 0.0)
        bonuses     = 1.0 + WASTE_BONUS_MAX * waste_ratio

        base_scores = WEIGHT_COVERAGE * cov_scores + WEIGHT_TFIDF * tfidf_sim
        MAX_RAW = WEIGHT_COVERAGE * 1.8 + WEIGHT_TFIDF * 1.0
        base_scores_norm = np.clip(base_scores / MAX_RAW, 0.0, 1.0)

        final_scores  = np.round(base_scores_norm * penalties * bonuses, 4)
        rating_scores = np.round(final_scores * (1 + RATING_BOOST * ratings / RATING_MAX), 4)
        almost_done   = (totals > 0) & (fulls / np.maximum(totals, 1) >= ALMOST_DONE_RATIO)
        voldoende_dekking = (fulls >= 1) | (partials >= 2)

        df = pd.DataFrame({
            "recipe_id":         self.recipes["ID"].values,
            "recipe_name":       self.recipes["Naam"].values,
            "category":          self.recipes["category"].values,
            "score_final":       final_scores,
            "score_with_rating": rating_scores,
            "waste_bonus":       np.round(bonuses, 3),
            "length_penalty":    np.round(penalties, 4),
            "coverage":          np.round(base_scores_norm, 4),
            "tfidf":             np.round(tfidf_sim, 4),
            "full":              fulls,
            "partial":           partials,
            "total":             totals,
            "urgent_items":      urgents,
            "n_ingredients":     n_ing,
            "almost_done":       almost_done,
            "url":               self.recipes["URL"].fillna("").values,
            "rating":            ratings,
        })

        df = df[(df["full"] >= 1) | (df["partial"] >= 2)]

        if only_almost_done:
            df = df[df["almost_done"]]

        if df.empty:
            return {"match": pd.DataFrame(), "with_rating": pd.DataFrame()}

        df = df.sort_values("score_final", ascending=False).reset_index(drop=True)

        top_match  = cap_categories(mmr_rerank(df.head(50), self.recipe_vecs, self.recipe_index, top_n * 2), top_n)
        top_rating = cap_categories(mmr_rerank(df.sort_values("score_with_rating", ascending=False).head(50),
                                               self.recipe_vecs, self.recipe_index, 6,
                                               score_col="score_with_rating"), 3)
        top_match["rank"]  = top_match.index + 1
        top_rating["rank"] = top_rating.index + 1

        if verbose:
            self._print(user_id, profile, top_match, top_rating, pantry)

        return {"match": top_match, "with_rating": top_rating}

    def _print(self, user_id, profile, top_match, top_rating, pantry):
        def show(label, df, col):
            print(f"\n{'-'*55}\n  {label}\n{'-'*55}")
            for _, r in df.iterrows():
                flags = ("  ✓ bijna klaar" if r["almost_done"] else "") + \
                        (f"  🔴 {r['urgent_items']} urgent" if r["urgent_items"] else "") + \
                        (f"  ⚠ penalty={r['length_penalty']}" if r["length_penalty"] < 1 else "")
                print(f"\n  #{r['rank']:02d} [{r['category']}] {r['recipe_name']}{flags}")
                print(f"       score={r[col]:.3f}  dekking={r['coverage']:.3f}  tfidf={r['tfidf']:.3f}  rating={r['rating']:.1f}")
                print(f"       {r['full']}/{r['total']} volledig, {r['partial']} gedeeltelijk")
                print(f"       {r['url']}")
        print(f"\n{'='*55}\n  Gebruiker {user_id}  |  {profile}\n{'='*55}")
        show("TOP 3 - match + rating", top_rating, "score_with_rating")
        show(f"TOP {len(top_match)} - beste match", top_match, "score_final")

    def recommend_all(self, top_n=10):
        all_dfs = []
        print(f"\nRun voor {len(self.users)} gebruikers...")
        for i, uid in enumerate(self.users, 1):
            print(f"  [{i}/{len(self.users)}] gebruiker {uid}", flush=True)
            df = self.recommend(uid, top_n=top_n)["match"]
            if df.empty: continue
            profile = self.pantry[self.pantry["user_id"] == uid]["profiel"].iloc[0]
            df.insert(0, "user_id", uid); df.insert(1, "profile", profile)
            all_dfs.append(df)
        return pd.concat(all_dfs, ignore_index=True)

    def export(self, df, path):
        from openpyxl.styles import Alignment, Font, PatternFill
        green = PatternFill(start_color="CCFFCC", end_color="CCFFCC", fill_type="solid")
        sheet = "Aanbevelingen"
        with pd.ExcelWriter(path, engine="openpyxl") as writer:
            df.to_excel(writer, index=False, sheet_name=sheet)
            ws = writer.sheets[sheet]
            for cell in ws[1]: cell.font = Font(bold=True)
            almost_col = list(df.columns).index("almost_done") if "almost_done" in df.columns else -1
            for row in ws.iter_rows(min_row=2):
                for cell in row: cell.alignment = Alignment(wrap_text=True, vertical="top")
                if almost_col >= 0 and row[almost_col].value:
                    for cell in row: cell.fill = green
        print(f"Opgeslagen: {path}")


print("WasteLessRecommender klasse geladen")

WasteLessRecommender klasse geladen


## 7. Initialiseer de aanbeveler
> Zorg dat `recepten_AH.xlsx` en `pantry_proxies.xlsx` in dezelfde map staan als dit notebook.

In [7]:
rec = WasteLessRecommender()

Data laden... [OK] (2000 recepten, 100 gebruikers)
[OK]pten parsen... 
[OK]DF bouwen... 


## 8. Aanbevelingen voor één gebruiker

In [8]:
# Pas USER_ID en TOP_N aan naar wens
USER_ID = 1
TOP_N   = 5

result = rec.recommend(USER_ID, top_n=TOP_N, verbose=True)


  Gebruiker 1  |  stel

-------------------------------------------------------
  TOP 3 - match + rating
-------------------------------------------------------

  #01 [overig] Ananas met honing en tijm van de BBQ van Hugo Kennis  ✓ bijna klaar  ⚠ penalty=0.9333
       score=0.419  dekking=0.384  tfidf=0.057  rating=8.4
       4/5 volledig, 0 gedeeltelijk
       https://www.ah.nl/allerhande/recept/R-R1200073/

  #02 [overig] Zeebaarsfilet van de BBQ  ✓ bijna klaar  ⚠ penalty=0.9333
       score=0.412  dekking=0.377  tfidf=0.014  rating=8.5
       4/5 volledig, 0 gedeeltelijk
       https://www.ah.nl/allerhande/recept/R-R1201126/

  #03 [dessert] Courgette-citroenmuffins  ✓ bijna klaar
       score=0.394  dekking=0.337  tfidf=0.058  rating=8.4
       7/10 volledig, 0 gedeeltelijk
       https://www.ah.nl/allerhande/recept/R-R1201221/

-------------------------------------------------------
  TOP 5 - beste match
-------------------------------------------------------

  #01 [overig] Ana

### Bekijk resultaten als DataFrame

In [9]:
# Beste match
result["match"][["rank", "recipe_name", "category", "score_final", "coverage", "full", "total", "urgent_items", "url"]]

,rank,recipe_name,category,score_final,coverage,full,total,urgent_items,url
0,1,Ananas met honing en tijm van de BBQ van Hugo ...,overig,0.3584,0.3840,4,5,0,https://www.ah.nl/allerhande/recept/R-R1200073/
1,2,Zeebaarsfilet van de BBQ,overig,0.3521,0.3772,4,5,0,https://www.ah.nl/allerhande/recept/R-R1201126/
2,3,Tortilla's met kip en groente van Giorgia's Ki...,vlees,0.3415,0.3415,7,11,0,https://www.ah.nl/allerhande/recept/R-R1201415/
3,4,Courgette-citroenmuffins,dessert,0.3371,0.3371,7,10,0,https://www.ah.nl/allerhande/recept/R-R1201221/
4,5,Wafels van leftover friet,ontbijt,0.3143,0.3143,4,6,0,https://www.ah.nl/allerhande/recept/R-R1199811/


In [10]:
# Match + rating
result["with_rating"][["rank", "recipe_name", "category", "score_with_rating", "rating", "url"]]

,rank,recipe_name,category,score_with_rating,rating,url
0,1,Ananas met honing en tijm van de BBQ van Hugo ...,overig,0.4186,8.4,https://www.ah.nl/allerhande/recept/R-R1200073/
1,2,Zeebaarsfilet van de BBQ,overig,0.4120,8.5,https://www.ah.nl/allerhande/recept/R-R1201126/
2,3,Courgette-citroenmuffins,dessert,0.3937,8.4,https://www.ah.nl/allerhande/recept/R-R1201221/


## 9. Alleen 'bijna klaar' recepten

In [11]:
result_almost = rec.recommend(USER_ID, top_n=TOP_N, only_almost_done=True, verbose=True)

if result_almost["match"].empty:
    print("Geen recepten gevonden waarvoor je bijna alle ingrediënten hebt.")
else:
    display(result_almost["match"][["rank", "recipe_name", "score_final", "full", "total", "url"]])


  Gebruiker 1  |  stel

-------------------------------------------------------
  TOP 3 - match + rating
-------------------------------------------------------

  #01 [overig] Ananas met honing en tijm van de BBQ van Hugo Kennis  ✓ bijna klaar  ⚠ penalty=0.9333
       score=0.419  dekking=0.384  tfidf=0.057  rating=8.4
       4/5 volledig, 0 gedeeltelijk
       https://www.ah.nl/allerhande/recept/R-R1200073/

  #02 [overig] Zeebaarsfilet van de BBQ  ✓ bijna klaar  ⚠ penalty=0.9333
       score=0.412  dekking=0.377  tfidf=0.014  rating=8.5
       4/5 volledig, 0 gedeeltelijk
       https://www.ah.nl/allerhande/recept/R-R1201126/

  #03 [dessert] Courgette-citroenmuffins  ✓ bijna klaar
       score=0.394  dekking=0.337  tfidf=0.058  rating=8.4
       7/10 volledig, 0 gedeeltelijk
       https://www.ah.nl/allerhande/recept/R-R1201221/

-------------------------------------------------------
  TOP 5 - beste match
-------------------------------------------------------

  #01 [overig] Ana

,rank,recipe_name,score_final,full,total,url
0,1,Ananas met honing en tijm van de BBQ van Hugo ...,0.3584,4,5,https://www.ah.nl/allerhande/recept/R-R1200073/
1,2,Zeebaarsfilet van de BBQ,0.3521,4,5,https://www.ah.nl/allerhande/recept/R-R1201126/
2,3,Tortilla's met kip en groente van Giorgia's Ki...,0.3415,7,11,https://www.ah.nl/allerhande/recept/R-R1201415/
3,4,Courgette-citroenmuffins,0.3371,7,10,https://www.ah.nl/allerhande/recept/R-R1201221/
4,5,Wafels van leftover friet,0.3143,4,6,https://www.ah.nl/allerhande/recept/R-R1199811/


## 10. Alle gebruikers → export naar Excel

In [12]:
# Pas het bestandsnaam en aantal resultaten aan naar wens
OUTPUT_FILE = "aanbevelingen.xlsx"
TOP_N_ALL   = 10

all_results = rec.recommend_all(top_n=TOP_N_ALL)
rec.export(all_results, OUTPUT_FILE)

print(f"\nTotaal: {len(all_results)} aanbevelingen voor {all_results['user_id'].nunique()} gebruikers")
all_results.head()


Run voor 100 gebruikers...
  [1/100] gebruiker 1
  [2/100] gebruiker 2
  [3/100] gebruiker 3
  [4/100] gebruiker 4
  [5/100] gebruiker 5
  [6/100] gebruiker 6
  [7/100] gebruiker 7
  [8/100] gebruiker 8
  [9/100] gebruiker 9
  [10/100] gebruiker 10
  [11/100] gebruiker 11
  [12/100] gebruiker 12
  [13/100] gebruiker 13
  [14/100] gebruiker 14
  [15/100] gebruiker 15
  [16/100] gebruiker 16
  [17/100] gebruiker 17
  [18/100] gebruiker 18
  [19/100] gebruiker 19
  [20/100] gebruiker 20
  [21/100] gebruiker 21
  [22/100] gebruiker 22
  [23/100] gebruiker 23
  [24/100] gebruiker 24
  [25/100] gebruiker 25
  [26/100] gebruiker 26
  [27/100] gebruiker 27
  [28/100] gebruiker 28
  [29/100] gebruiker 29
  [30/100] gebruiker 30
  [31/100] gebruiker 31
  [32/100] gebruiker 32
  [33/100] gebruiker 33
  [34/100] gebruiker 34
  [35/100] gebruiker 35
  [36/100] gebruiker 36
  [37/100] gebruiker 37
  [38/100] gebruiker 38
  [39/100] gebruiker 39
  [40/100] gebruiker 40
  [41/100] gebruiker 41
  [42/

,user_id,profile,recipe_id,recipe_name,category,score_final,score_with_rating,waste_bonus,length_penalty,coverage,tfidf,full,partial,total,urgent_items,n_ingredients,almost_done,url,rating,rank
0,1,stel,1200073,Ananas met honing en tijm van de BBQ van Hugo ...,overig,0.3584,0.4186,1.0,0.9333,0.3840,0.0573,4,0,5,0,5,True,https://www.ah.nl/allerhande/recept/R-R1200073/,8.4,1
1,1,stel,1201126,Zeebaarsfilet van de BBQ,overig,0.3521,0.4120,1.0,0.9333,0.3772,0.0142,4,0,5,0,5,True,https://www.ah.nl/allerhande/recept/R-R1201126/,8.5,2
2,1,stel,1201415,Tortilla's met kip en groente van Giorgia's Ki...,vlees,0.3415,0.3791,1.0,1.0000,0.3415,0.0495,7,1,11,0,11,True,https://www.ah.nl/allerhande/recept/R-R1201415/,5.5,3
3,1,stel,1201221,Courgette-citroenmuffins,dessert,0.3371,0.3937,1.0,1.0000,0.3371,0.0577,7,0,10,0,10,True,https://www.ah.nl/allerhande/recept/R-R1201221/,8.4,4
4,1,stel,1199811,Wafels van leftover friet,ontbijt,0.3143,0.3470,1.0,1.0000,0.3143,0.0117,4,0,6,0,6,True,https://www.ah.nl/allerhande/recept/R-R1199811/,5.2,5


# Sensitiviteitsanalyse voor Discussie

In [13]:
weight_configs = [
    (1.00, 0.00),
    (0.75, 0.25),
    (0.50, 0.50),
    (0.25, 0.75),
    (0.00, 1.00),
]

resultaten = []
SAMPLE_USERS = rec.users[:5]

for w_cov, w_tfidf in weight_configs:
    globals()["WEIGHT_COVERAGE"] = w_cov
    globals()["WEIGHT_TFIDF"]    = w_tfidf

    alle = []
    for uid in SAMPLE_USERS:
        res = rec.recommend(uid, top_n=10)
        if not res["match"].empty:
            df = res["match"].copy()
            df["user_id"] = uid
            alle.append(df)

    all_results = pd.concat(alle).reset_index(drop=True)
    top3  = all_results[all_results["rank"] <= 3]
    rank1 = all_results[all_results["rank"] == 1]

    resultaten.append({
        "coverage_weight":       w_cov,
        "tfidf_weight":          w_tfidf,
        "gem_coverage_top3":     round(top3["coverage"].mean(), 3),
        "pct_almost_ready_top3": round(top3["almost_done"].mean() * 100, 1),
        "gem_score_rank1":       round(rank1["score_final"].mean(), 3),
    })

WEIGHT_COVERAGE = 0.75
WEIGHT_TFIDF    = 0.25

df_sensitivity = pd.DataFrame(resultaten)
display(df_sensitivity)

,coverage_weight,tfidf_weight,gem_coverage_top3,pct_almost_ready_top3,gem_score_rank1
0,1.00,0.00,0.374,73.3,0.381
1,0.75,0.25,0.321,66.7,0.329
2,0.50,0.50,0.255,73.3,0.264
3,0.25,0.75,0.171,73.3,0.180
4,0.00,1.00,0.072,20.0,0.088


De almost-ready rate blijft stabiel rond 73% voor alle coverage-dominante 
configuraties. Alleen bij volledige TF-IDF dominantie (0.00/1.00) zakt deze naar 
20%, wat bevestigt dat pantry coverage het essentiële signaal is voor het 
waste-reductiedoel. De gekozen 0.75/0.25 verdeling is daarmee een robuuste keuze 
binnen een breed bereik van coverage-dominante instellingen.